# The Continuum population model: a guide with live figures

Continuum fills a blank Cardano devnet with simulated users who are born, act, go quiet and leave, at random but realistic rates. This notebook mirrors the beginner guide section by section. Drag the sliders; the pictures redraw.

Every code cell is one or two calls into `guide_lib.py` (toy models and data) and `guide_figures.py` (drawing); open those files to see how a picture is made. The cells marked **For the curious** give the formulas behind a section. They are optional: nothing later depends on having read them.

Where the guide used illustrations, this notebook uses the **real preprod and preview fits** shipped in `profiles/`, and (when `/data/chain` and `/data/export` are mounted, see `make lab`) the real preprod chain and the seed-7 run. Without them it falls back to tiny test fixtures and says so below. Looking for the calibration workflow itself? See [explore.ipynb](explore.ipynb).

Free companions: [Seeing Theory](https://seeing-theory.brown.edu/) (a visual course in probability), [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html), and [scikit-learn mixtures](https://scikit-learn.org/stable/modules/mixture.html).

**Sections:** 1 Two layers of randomness · 2 Lifecycles and hazard rates · 3 The four distributions · 4 When things happen · 5 Seeds and replay · 6 Calibration · 7 Clustering · 8 Profiles per network · 9 Currencies and exchanges · 10 Humans and their wallets · 11 Glossary

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())
sys.path.insert(0, '/work/lab')
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
import guide_lib as gl
import guide_figures as gf

PATHS = gl.data_paths()
NETS = gl.networks()
print('data:', PATHS.describe())
print('networks:', {net.name: ('fitted' if net.fitted else 'not fitted yet') for net in NETS})

## 1. Two layers of randomness

Continuum is an *agent-based, discrete-event* simulation: a list of individual actors and a queue of "things that happen next", each with a timestamp. Randomness enters at two speeds, kept apart so "more users" and "busier users" can be turned independently.

- **Population layer** (days to weeks): births, dormancy, retirement. Driven by *hazard rates*.
- **Activity layer** (seconds to hours): wake-ups, a Poisson process with a daily cycle and bursts. An awake actor's agenda then picks an intent (pay, wait, ...).

Nothing changes an actor until the chain confirms it. Below, the bands are slow awake/dormant spells and the ticks are fast wake-ups, for six actors over ten days.

Learn more: [Agent-based model](https://en.wikipedia.org/wiki/Agent-based_model) · [Discrete-event simulation](https://en.wikipedia.org/wiki/Discrete-event_simulation) · [Seeing Theory](https://seeing-theory.brown.edu/)

In [ ]:
interact(gf.show_two_layers, seed=IntSlider(3, 0, 20, description='seed'));

## 2. Lifecycles and hazard rates

A wallet is *Born*, *Active*, *Dormant* (wakes after a log-normal sleep) or *Abandoned*. "When does a wallet go quiet?" is answered by a **hazard rate**: the chance per unit time that it happens now, given it has not happened yet. A constant hazard of one per day gives awake spells of one day *on average*, but any single spell may last an hour or a week: spell lengths are exponential.

Pools have their own cycle (Registered, Active, Retiring, Retired), gated on the chain: the simulator never assumes the chain did something. In the profile files the fields are `hazards.dormancy` and `hazards.dormancy_onset`.

Learn more: [Survival analysis](https://en.wikipedia.org/wiki/Survival_analysis) · [Failure rate (hazard)](https://en.wikipedia.org/wiki/Failure_rate) · [Finite-state machine](https://en.wikipedia.org/wiki/Finite-state_machine) · [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html)

**For the curious: the hazard and the exponential.** A constant hazard $h$ means "in the next instant $dt$, the event happens with probability $h\,dt$, no matter how long we have waited". The waiting time $T$ until it happens is then exponential:

$$P(T > t) = e^{-ht}, \qquad f(t) = h\,e^{-ht}, \qquad \mathbb{E}[T] = \frac{1}{h}$$

The red curve in the right-hand panel above is $f(t)$ with $h$ = `hazard/day`. An actor that alternates awake spells of mean $1/h$ with sleeps of mean $D$ is awake a share $\frac{1/h}{1/h + D}$ of the time; `population_fit.dormancy_onset` solves that for $h$.

In [ ]:
interact(gf.show_lifecycle,
         onset_per_day=FloatSlider(0.5, min=0.1, max=3, step=0.1, description='hazard/day'),
         sleep_median_days=FloatSlider(2, min=0.2, max=10, step=0.2, description='sleep (d)'),
         seed=IntSlider(1, 0, 20));

## 3. The four distributions

A profile is written as distributions, for example `competence: beta(8, 2)`. Each actor draws its own value once, so 1,000 wallets get 1,000 different but realistically spread personalities.

- **Beta**: a percentage-like trait between 0 and 1 (`competence`, `care`). Think of `a` as past successes and `b` as past failures.
- **Log-normal**: durations and amounts. Its logarithm is a bell curve, so it is positive and skewed: `hazards.dormancy`, the body of payment amounts.
- **Pareto**: heavy tails, "a few do most of it" (`rates.activity_tail`, `traits.patience`, payment tails). The smaller alpha, the heavier the tail.
- **Exponential**: the wait until the next event at a constant rate, with no memory. What a hazard produces, and what sits between Poisson ticks.

Learn more: [Beta](https://en.wikipedia.org/wiki/Beta_distribution) · [Log-normal](https://en.wikipedia.org/wiki/Log-normal_distribution) · [Pareto](https://en.wikipedia.org/wiki/Pareto_distribution) · [Exponential](https://en.wikipedia.org/wiki/Exponential_distribution) · [Heavy-tailed](https://en.wikipedia.org/wiki/Heavy-tailed_distribution) · [Seeing Theory: probability distributions](https://seeing-theory.brown.edu/probability-distributions/index.html) · [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html)

**For the curious: the four densities.** A density $f(x)$ is the curve drawn below: the area under it between two values is the probability of landing between them.

| distribution | density | where | mean |
|---|---|---|---|
| Beta$(a, b)$ | $f(x) = \dfrac{x^{a-1}(1-x)^{b-1}}{B(a,b)}$ | $0 < x < 1$ | $\dfrac{a}{a+b}$ |
| Log-normal$(\mu, \sigma)$ | $f(x) = \dfrac{1}{x\,\sigma\sqrt{2\pi}}\exp\!\left(-\dfrac{(\ln x-\mu)^2}{2\sigma^2}\right)$ | $x > 0$ | $e^{\mu+\sigma^2/2}$ |
| Pareto$(x_m, \alpha)$ | $f(x) = \dfrac{\alpha\,x_m^{\alpha}}{x^{\alpha+1}}$ | $x \ge x_m$ | $\dfrac{\alpha\,x_m}{\alpha-1}$ if $\alpha > 1$, else infinite |
| Exponential$(\lambda)$ | $f(t) = \lambda\,e^{-\lambda t}$ | $t \ge 0$ | $\dfrac{1}{\lambda}$ |

$B(a,b)$ is a constant that makes the area 1. Two readings worth keeping: a log-normal is a bell curve in $\ln x$, so its median is $e^{\mu}$; and a Pareto is easiest through its tail, $P(X > x) = (x_m/x)^{\alpha}$, a straight line on log-log axes.

In [ ]:
ON_RELEASE = dict(continuous_update=False)   # redraw when the slider is let go, not while it moves
interact(gf.show_four_distributions,
         a=FloatSlider(8, min=0.5, max=20, step=0.5, **ON_RELEASE), b=FloatSlider(2, min=0.5, max=20, step=0.5, **ON_RELEASE),
         mu=FloatSlider(1, min=-1, max=2.5, step=0.1, **ON_RELEASE), sigma=FloatSlider(0.5, min=0.1, max=1.5, step=0.05, **ON_RELEASE),
         alpha=FloatSlider(1.3, min=0.5, max=4, step=0.1, **ON_RELEASE), xm=FloatSlider(1, min=0.5, max=3, step=0.25, **ON_RELEASE),
         rate=FloatSlider(1, min=0.2, max=4, step=0.2, **ON_RELEASE));

## 4. When things happen: Poisson, cycles and bursts

An awake actor wakes at random times. The baseline is a **Poisson process**: a steady average rate with exponential gaps. Real usage is not steady, so two effects stack on top:

- **Daily cycle** multiplies the rate up and down (`1 + amplitude * sin(2 pi t / 24h)`).
- **Hawkes bursts**: every event adds `alpha` to the rate, and the boost decays at speed `beta`, so activity clumps like an NFT drop. Continuum requires `alpha < beta`; the ratio `alpha/beta` is the average number of follow-ups an event triggers, and at 1 or more bursts never die out.

A wake-up is not yet a transaction: the actor's agenda usually decides to stay idle. The figure shows the same seed with the layers you switch on.

Learn more: [Poisson point process](https://en.wikipedia.org/wiki/Poisson_point_process) · [Hawkes process](https://en.wikipedia.org/wiki/Hawkes_process) · [Thinning](https://en.wikipedia.org/wiki/Thinning_(statistics)) · [Seeing Theory](https://seeing-theory.brown.edu/) · [scipy.stats.poisson](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.poisson.html)

**For the curious: the intensity.** The blue line above is the intensity $\lambda(t)$, the expected number of wake-ups per hour at time $t$:

$$\lambda(t) = \underbrace{r\left(1 + A\sin\frac{2\pi t}{24\,\mathrm{h}}\right)}_{\text{base rate with daily cycle}} \;+\; \underbrace{\alpha\sum_{t_i < t} e^{-\beta\,(t - t_i)}}_{\text{Hawkes bursts}}$$

where $r$ is `rate`, $A$ is `daily`, and the sum runs over the earlier wake-ups $t_i$. Each wake-up adds $\alpha$ to the rate and the boost dies away at speed $\beta$; over its lifetime it causes $\int_0^\infty \alpha e^{-\beta s}\,ds = \alpha/\beta$ further wake-ups on average, which is why $\alpha < \beta$ is required. With $A = 0$ and $\alpha = 0$ this is a plain Poisson process: gaps between wake-ups are Exponential$(r)$, and the count in a stretch of length $T$ is Poisson with mean $rT$. `guide_lib.arrivals` simulates it.

In [ ]:
interact(gf.show_arrivals,
         rate=FloatSlider(1, min=0.2, max=4, step=0.2, **ON_RELEASE), daily=FloatSlider(0, min=0, max=0.9, step=0.1, **ON_RELEASE),
         alpha=FloatSlider(0, min=0, max=3, step=0.1, **ON_RELEASE), beta=FloatSlider(2, min=0.5, max=5, step=0.1, **ON_RELEASE),
         seed=IntSlider(4, 0, 30));

## 5. Seeds and replay: random, yet repeatable

A computer's random numbers are a deterministic sequence fixed by a starting **seed**. Continuum takes one root seed per run and splits it into an independent stream per path (`actor/7/activity`, a pool's KES key, ...) with a key-derivation function, **HKDF**. Two consequences the whole project relies on:

1. **Same seed, same run.** A bug found on Tuesday replays byte for byte on Friday.
2. **Adding an actor changes nobody else.** Actor 31 has its own stream, so actors 1 to 30 behave as before. That keeps bisecting across node builds meaningful.

The cell below derives streams the same way (HKDF-SHA256 over the path; the engine feeds ChaCha8, this toy uses PCG64) and checks both properties.

Learn more: [Pseudorandom number generator](https://en.wikipedia.org/wiki/Pseudorandom_number_generator) · [HKDF](https://en.wikipedia.org/wiki/HKDF) · [Random seed](https://en.wikipedia.org/wiki/Random_seed) · [numpy Generator](https://numpy.org/doc/stable/reference/random/generator.html)

In [ ]:
interact(gl.print_replay, seed=IntSlider(7, 0, 99, description='root seed'), actors=IntSlider(5, 2, 12));

## 6. Calibration: fit, run, grade

Made-up numbers make a made-up chain. So the lab measures real behaviour from a converted chain (`mithril2pq` Parquet), **fits** profile settings (`fit.py`), **runs** the simulation (`continuum run --fake`, `continuum export`), and **grades** the simulated chain with the same statistics (`grade.py`) against the saved reference (`target.json`). Where they disagree, the profile is wrong.

Fitting picks parameters that match data: maximum likelihood for value distributions, method of moments for the beta traits. Each estimator is a short function in `estimators.py` with its formula and what can go wrong; `fit.py` and `grade.py` read top to bottom as this loop.

**Grading with the KS statistic.** Plot each sample as a cumulative curve (the share at or below x). **D** is the largest vertical gap between the two curves: 0 is identical, 1 is disjoint. A statistic **passes when D <= 0.1**. Why not a p-value? With thousands of samples a p-value calls even a trivial difference significant; D measures how big the difference is, which is what matters. Try it: slide the shift and the sample size.

Learn more: [Empirical distribution function](https://en.wikipedia.org/wiki/Empirical_distribution_function) · [Kolmogorov-Smirnov test](https://en.wikipedia.org/wiki/Kolmogorov%E2%80%93Smirnov_test) · [Maximum likelihood](https://en.wikipedia.org/wiki/Maximum_likelihood_estimation) · [Effect size](https://en.wikipedia.org/wiki/Effect_size) · [scipy.stats.ks_2samp](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ks_2samp.html) · [Seeing Theory](https://seeing-theory.brown.edu/)

**For the curious: the KS statistic.** Write $F_{\text{run}}(x)$ for the share of the run's sample at or below $x$ (its empirical cumulative curve) and $F_{\text{target}}(x)$ for the target's. Then

$$D = \max_x \,\bigl|\,F_{\text{run}}(x) - F_{\text{target}}(x)\,\bigr|, \qquad \text{pass when } D \le 0.1$$

$D$ is a share, between 0 and 1: $D = 0.1$ means that at the worst point the two samples disagree about ten percentage points of their values. Two samples of 1,000 from the *same* distribution have $D \approx 0.04$ from noise alone, and a bell curve shifted by a quarter of its standard deviation has $D = 0.1$. The bound and the reasoning are in `compare.py` (`KS_PASS_DISTANCE`).

The transaction mix is compared by total variation distance, $TV = \tfrac{1}{2}\sum_c |a_c - b_c|$ over the categories $c$: the share of transactions that would have to change kind.

In [ ]:
interact(gf.show_ks_demo,
         shift=FloatSlider(0.15, min=0, max=1, step=0.05, **ON_RELEASE),
         n=IntSlider(2000, 100, 20000, step=100, **ON_RELEASE));

### Real data: the seed-7 run against the preprod target

`continuum run --fake --seed 7 --wallets 100 --epochs 2 --profiles preprod` simulates 100 wallets over 90 days. Each panel overlays the run's samples on the preprod target. Wherever the curves part by more than 0.1, the profile needs work; the known gaps are the engine's 2 ADA payment floor, gaps that are exponential where the chain's are bursty, and a type mix that is all payments.

In [ ]:
run_samples, overall_target, target_path = gl.run_and_target(PATHS)
print('run:', PATHS.export, '| target:', os.path.basename(target_path), '| real run' if PATHS.real else '| fixture run (not the seed-7 preprod run)')
gf.fig_ks(run_samples, overall_target); plt.show()

## 7. Clustering: many kinds of user, not one average

One profile fitted to everyone describes an "average user" who does not exist. Clustering first sorts units into groups that behave alike, then fits a profile per group.

- **What is one "user"?** A wallet spreads its funds over many addresses that share one *stake credential*, so a **unit** is the stake credential where there is one and the address otherwise.
- **Features** per unit (documented column by column in `unit_features.py`): transactions per day, median gap, median payment, in/out counts, active days, longest dormancy (log-scaled, then standardised). Units that mostly call scripts or send staking certificates are split off first by absolute thresholds, because shares pinned at exactly 0 or 1 make a Gaussian mixture chase tiny-variance components.
- **Gaussian mixture**: assume the cloud of units is overlapping blobs and find them. **BIC** scores a fit by how well it explains the data minus a penalty for each extra blob; the lab searches k = 2..15 and takes the elbow of the curve.
- **Weighting**: by number of units gives a realistic population; by number of transactions gives realistic traffic. A few busy bots make the two differ.

Cluster names come from absolute thresholds in real units (see `lab/README.md`): `holder` below 0.05 tx/day, `occasional-payer` up to 1 tx/day, `frequent-trader` above, `script-user` and `staker` when at least half the unit's transactions are script calls or staking certificates; a size suffix (`-small` under 10 ADA, `-large` from 1,000 ADA median payment) separates repeats.

Learn more: [Cluster analysis](https://en.wikipedia.org/wiki/Cluster_analysis) · [Mixture model](https://en.wikipedia.org/wiki/Mixture_model) · [Bayesian information criterion](https://en.wikipedia.org/wiki/Bayesian_information_criterion) · [scikit-learn: Gaussian mixtures](https://scikit-learn.org/stable/modules/mixture.html) · [Seeing Theory](https://seeing-theory.brown.edu/)

**For the curious: the mixture and BIC.** A Gaussian mixture with $k$ components says the density of a unit with feature vector $x$ is a weighted sum of bell-shaped blobs:

$$p(x) = \sum_{j=1}^{k} w_j\,\mathcal{N}(x;\,\mu_j, \Sigma_j), \qquad \sum_j w_j = 1$$

Blob $j$ has weight $w_j$ (its share of the units), centre $\mu_j$ and spread $\Sigma_j$. The fit chooses them to maximise the likelihood of the $n$ observed units,

$$L = \prod_{i=1}^{n} p(x_i)$$

More blobs always raise $L$, so $k$ is chosen with the Bayesian information criterion, which charges for each of the $p$ free parameters:

$$\mathrm{BIC} = p\,\ln n \;-\; 2\ln L \qquad \text{(lower is better)}$$

The lab uses diagonal $\Sigma_j$ (one variance per feature, $p = 2dk + k - 1$ for $d$ features) and adds `REG_COVAR = 0.1` to every variance so no blob can collapse onto a spike. On real chains BIC keeps falling slowly, so $k$ is taken at the **elbow**: the first $k$ whose next step gains less than 5% of the whole drop. `cluster.py` has the details and how sensitive the result is to each constant.

In [ ]:
gl.print_one_wallet_one_unit()

In [ ]:
units_table = gl.units_table(PATHS)                      # one row per unit: unit_features.COLUMNS
clustering = gl.cluster_units_table(units_table, PATHS)  # adds the `cluster` column
print(gl.describe_clustering(units_table, clustering))
print(gl.cluster_summary(units_table))

In [ ]:
gf.fig_units_scatter(units_table, PATHS.real).show()

In [ ]:
gf.fig_weighting(*gl.cluster_weights(units_table)); plt.show()
gf.fig_bic(*gf.bic_curves(clustering.fit, NETS[:2])); plt.show()

## 8. What each network looks like

The fitted profile files in `profiles/` hold one profile per cluster, each with a weight; `continuum run --profiles preprod` splits wallets by those weights. Below, the preview and preprod fits side by side, then the same views for mainnet, which shows **not fitted yet** until `profiles/mainnet.yaml` exists and fills itself in then.

Learn more: [Probability density function](https://en.wikipedia.org/wiki/Probability_density_function) · [Cumulative distribution function](https://en.wikipedia.org/wiki/Cumulative_distribution_function) · [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html)

In [ ]:
for net in NETS:
    print(net.summary())
gf.fig_shares(NETS); plt.show()

In [ ]:
gf.fig_quantile_cdfs(NETS, 'activity_per_day', 'transactions per day per unit', 'Activity (per cluster)', logx=True); plt.show()
gf.fig_quantile_cdfs(NETS, 'log10_gap_seconds', 'log10 seconds between a unit\'s transactions', 'Gaps between transactions'); plt.show()

In [ ]:
gf.fig_amounts(NETS); plt.show()
gf.fig_mix(NETS); plt.show()
gf.fig_traits(NETS); plt.show()

## 9. Currencies and exchanges

A run with a `markets:` section in its profile file (the default `m1` has one) gives its wallets an ADA market. The price of ADA is a **rate process**: ADA/USD takes a random step every ten simulated seconds, its drift flipping between a bull and a bear regime, with occasional downward-leaning jumps; each currency's USD rate wanders around a level that trends at its own drift, with its own volatility and pull back to the trend, all fitted to a year of real rates (a committed snapshot, regenerated with `make market-snapshot`; the engine never fetches). ADA in euros is ADA/USD divided by what a euro buys in dollars. The path is a function of the run's seed and the clock alone: what the simulated wallets do never moves it, so replay recomputes it rather than storing it.

Wallets buy and sell through **exchanges**. Each quotes a **bid** (what it pays for ADA) and an **ask** (what it charges) around the mid rate: the gap is its **spread**, and a slow drift of its own, the **basis**, makes two exchanges quote the same currency slightly differently. The rates are compressed on purpose: a fake run lasts minutes, so the defaults pack a market's swings into hours. The left figure is one seeded path, the right the first hour of three exchanges' USD quotes.

The figures use a copy of the model in `guide_lib`, the same processes and parameters as `internal/market` drawn with numpy, so a path here matches a run's in shape, not draw for draw.


In [ ]:
interact(gf.show_rate_path, seed=IntSlider(4, 0, 20, description='seed'));

**Preferred currencies.** Each wallet has one **primary currency**, drawn by the currencies' population shares, and thinks in it: its cost basis and gains are in it. Most wallets have no other; each further currency comes with probability *q* (0.15 by default), so the count is geometric, $P(k) = (1-q)\,q^k$: a few have two currencies, very few three.


In [ ]:
interact(gf.show_extra_currencies, extra=FloatSlider(0.15, min=0.05, max=0.6, step=0.05, **ON_RELEASE));

**Buying and selling.** A wallet's agenda now and then scores `buy_ada` or `sell_ada` (the why box shows the scores and the position they read). It picks the exchange with the best effective price in its primary currency, among those quoting one of its currencies: a non-primary currency pays a conversion, the last exchange it used gets a small loyalty discount, and a little noise breaks near-ties. A buy is paid in fiat off chain; the exchange's hot wallet then pays the ADA on chain, less a withdrawal fee. A sale is the wallet paying ADA on chain to the exchange, which credits fiat. Fiat never touches the chain, so the engine's value check still accounts for every lovelace. Once the ADA is k deep the trade settles: a buy moves the wallet's **cost basis**, the average price it paid per ADA.

**Births and deaths are synthetic, by design.** In a market run, wallet births and, for profiles that opt in (`hazards.market: true`), dormancy and abandonment are not fitted from chain data: they are configurable parameters driven by the rate. Each currency's wallets are born at the profile's rate times the currency's share times a **birth factor**, $\min(\max(e^{\beta r}, \text{lo}), \text{hi})$, with $r$ the log return of ADA in that currency over five minutes: a rally against the euro brings euro wallets. A prolonged fall (half an hour) raises a **death factor** the same way, so wallets go quiet and leave faster; a hodl wallet takes its square root and resists. A wallet that abandons sells its ADA first.


In [ ]:
interact(gf.show_lifecycle_factors, seed=IntSlider(4, 0, 20, description='seed'));

**Sell or hodl.** In a **drawdown**, when the price falls below a wallet's cost basis by more than 10%, the traits decide. A wallet on the hodl agenda holds for the recovery whatever happens. Any other scores a sale that grows with its **loss aversion** (a new trait in [0, 1]), its impatience and the depth of the loss; above water it sells now and then, and past a 15% gain it takes profit. The figure draws m1's traders and holders and counts, at a given depth, who sold within twenty decisions. A fake run's summary prints the same split for its largest drawdown, and the Markets page shows it.


In [ ]:
interact(gf.show_drawdown_split, depth=FloatSlider(-0.3, min=-0.8, max=-0.1, step=0.05, **ON_RELEASE));

## 10. Humans and their wallets

Until now every simulated **wallet** (a set of keys and the coins they control) behaved as if it were a person of its own. Real people are not like that: one person often keeps several wallets. Continuum now has a separate idea, the **human**: a simulated person who *owns* one or more wallets.

What belongs to whom:

| belongs to the **human** | belongs to each **wallet** |
|---|---|
| its **motivation** (the agenda: hodl, yield, profit, civic, dapp or opportunist) | its keys, coins and transaction history |
| its temperament: competence, patience, care, loss aversion | its own activity level and clock (fitted per on-chain unit) |
| its money outside the chain (**fiat**: ordinary currency like euros), its pay and bills | its stake delegation and its DRep delegation |
| its **wealth tier**: low, mid, high or whale | its **purpose** |

A wallet's **purpose** is what its owner uses it for, and it limits what the wallet does:

| purpose | what the wallet may do |
|---|---|
| savings | hold coins, receive purchases, stake; never trades itself |
| spending | everyday payments and trades |
| staking | delegate stake and collect rewards |
| governance | register as a DRep, vote |
| trading | buy, sell, use apps |

When a wallet wakes up, it asks its owner: the owner's motivation scores the choices, and the wallet's purpose rules out what the wallet is not for. (You can see this in the UI: the Decisions page's *why* box names the owner and the purpose, and a ruled-out choice says, for example, "savings wallet: not used for send".)

**Why one person holds several wallets.** Habit and safety: savings kept apart from spending money, a separate wallet for each app, stake split over several pools. Richer people tend to keep more. Each human draws how many wallets it means to keep: one, plus a few extras whose average grows with its tier. The figure draws 2000 humans from the shipped tiers; the `scale` slider multiplies every tier's extras.


In [ ]:
interact(gf.show_wallets_per_human, seed=IntSlider(7, 0, 20, description='seed'),
         scale=FloatSlider(1.0, min=0.0, max=3.0, step=0.25, description='extras ×', **ON_RELEASE));

**Why this matters for reading chain data.** The chain sees wallets, never people. Tools that count "users" from on-chain data have to guess which addresses belong together, a technique called **address clustering** (for example: addresses whose coins are spent together in one transaction probably share an owner). With humans in the simulation, the guess can be checked: the run knows the true owner of every wallet. Two things follow:

* **Counting.** A run with 95 humans owning 144 wallets has fewer people than wallets. An analysis that treats each wallet as one person overcounts people and undercounts how much each one holds.
* **Delegation spread.** Each wallet delegates on its own, so one person's stake can sit on several pools. Each human draws a **spread** in [0, 1]: when one of its wallets chooses a pool, a pool its other wallets already use looks less attractive by a factor $(1-\text{spread})^n$, where $n$ is how many of its wallets are already there. A spread of 0 puts every wallet on the best pool; a spread of 1 avoids any pool already used. Seen from the chain, one high-spread whale looks like several modest delegators.

**Where the wallets come from.** In the fitted profiles (mainnet, preprod, preview) the wallets still appear at the fitted rates of their on-chain clusters; the human layer decides *whose* each new wallet is. A new wallet joins a living human with the same motivation who still wants another wallet, or else it founds a new human. So the chain's fitted behaviour stays close to the fit, and the people on top are synthetic.


### The household money loop

Each human has a **household**: money that comes in, money that must go out, and a plan for the rest. Every number here is **synthetic** (made up to be plausible, not measured from any data) and set in the profile's `humans:` section.

Once a **month** (300 simulated seconds by default, so a short run sees many paydays):

1. **Employment.** An employed human loses its job with a small chance (the tier's `layoff`, 2% a month for the mid tier); an unemployed one finds a job with a larger chance (`hire`, 30%). A spell out of work therefore lasts about 1 / 0.3 ≈ 3.3 months on average.
2. **Pay.** The human receives its monthly **income**, or, out of work, only a **benefit**, a share (30%) of it.
3. **Bills.** Its monthly **expenses** (rent, food, debts) are a share of its pay, drawn once per human. They are paid from its fiat. Fiat never goes below zero: if the bills are more than the fiat, the rest is unpaid, the human's wallets grow more frustrated (likelier to go quiet or leave), and a **forced sale** of ADA follows, from a spending or trading wallet (after moving ADA out of savings if it has to).
4. **Investing.** What is left after bills is the **disposable income**. The human puts a share of it (its **investment share**, 20% on average) into ADA, in small regular amounts: twice a month by default. Buying a fixed amount on a schedule, whatever the price, is called **dollar-cost averaging (DCA)**. The purchase goes through an exchange, which pays the ADA on chain into the human's savings wallet. Before investing, the human keeps a **buffer** of one month of bills in fiat.

A worked example for a typical mid-tier human, computed from the shipped defaults:


In [ ]:
gl.print_worked_example('mid', ada_usd=0.15)

Some humans time their purchases: a **dip** buyer buys more after ADA has fallen and less after it has risen; a **momentum** buyer does the reverse. Each motivation has a style: profit and opportunist humans chase momentum, dapp users buy dips, everyone else averages in.

Purchases are not the only buys: a wallet's own agenda still scores `buy_ada` now and then (a rally draws buyers), but only when its owner has fiat above the buffer. The **fiat ledger** keeps every cent: an order **holds** its fiat when it is placed, spends it when the ADA is final, and gives it back if the order fails. The fake run's summary checks that what the humans hold equals what they started with, plus pay and sales, minus bills and purchases (`fiat conserved: true`). A small formal model, `models/specs/fiat-escrow`, checked that rule before the code was written.

The figure follows a typical human of each tier for two years. Left: its fiat, measured in months of its own pay (so a euro earner and a rupee earner share one axis); crosses mark months out of work; the dashed line is the buffer. Right: what it has invested so far. Try a higher layoff chance: the low tier, whose bills take almost all its pay, stalls first.


In [ ]:
interact(gf.show_household,
         share=FloatSlider(0.2, min=0.0, max=0.6, step=0.05, description='invest share', **ON_RELEASE),
         layoff=FloatSlider(0.02, min=0.0, max=0.3, step=0.01, description='layoff/month', **ON_RELEASE),
         seed=IntSlider(3, 0, 20, description='seed'));

### The same thing in a real fake run

These charts come from a fake mainnet run (`continuum run --fake --seed 7 --wallets 100 --profiles mainnet --scale 200 --epochs 12`), summarised by `humans_run.py`. Set `GUIDE_RUN` to a run directory of your own (it needs the run's `events.log` and `store.db`) to draw yours instead.

* **Left:** each tier's median fiat in months of pay: savings build up month by month, fastest for the tiers whose bills leave more over.
* **Middle:** how many wallets each human owns.
* **Right:** ADA bought every five minutes from income (DCA, steady) and by the wallets' own agendas (in bursts), against the ADA price.


In [ ]:
summary, source = gl.humans_run()
print('from', source)
print(summary['humans'], 'humans; wallets per human:', summary['wallets_per_human'], '; tiers:', summary['tiers'])
gf.fig_run_humans(summary); plt.show()

### Try it

1. **Invest more.** Copy `profiles/m1.yaml`, raise `humans.invest.share` from `beta(2, 8)` (mean 20%) to `beta(4, 6)` (mean 40%), and run `go run ./cmd/continuum run --fake --seed 7 --profiles YOUR.yaml --epochs 3 --out run2`. Compare the summary's `household:` line (purchases from income) and `markets:` line (buys per exchange) with a run of the shipped file. Predict first: will the exchanges run dry more often? (Look for `no exchange with free ADA` in `skipped by reason`.)
2. **A recession.** Raise every tier's `layoff` to 0.15 and lower `hire` to 0.1. How does the share of paydays out of work change in the summary, and do `short of bills` and forced sales (`forced_sale` in the skip reasons) rise?
3. **Lone wallets.** Set every tier's `wallets` to 0: each human keeps one wallet. The `humans:` line shows the count; check that delegations are no longer spread.
4. In the notebook, move the `layoff/month` slider to 0.2 and find the tier whose fiat never reaches the buffer.

(`--profiles` takes a shipped set's name, such as `m1` or `mainnet`, or the path of a profile file.)


## 11. Glossary

| term | meaning |
|---|---|
| distribution | how likely each possible value is |
| PDF / density | the curve whose height shows where values are common |
| CDF | cumulative curve: the share of values at or below x; the KS test compares CDFs |
| hazard rate | chance per unit time that an event happens now, given it has not yet |
| Poisson process | events at a steady average rate with no memory; gaps are exponential |
| Hawkes process | a Poisson process where each event temporarily raises the rate, causing bursts (needs alpha < beta) |
| heavy tail | extreme values are rare but far more common than a bell curve predicts |
| seed | the starting number that fixes a pseudo-random sequence: same seed, same run |
| HKDF | key-derivation function that splits one seed into independent per-path streams |
| fit | choosing a distribution's parameters to match observed data |
| KS statistic D | the largest gap between two cumulative curves; passes at D <= 0.1 |
| unit | the stake credential of an address (the address when it has none): one "user" |
| cluster | a group of units that behave alike; each gets its own profile and weight |
| BIC | model score that rewards fit and penalises extra clusters; lower is better |
| k | the number of mixture components (clusters) |
| k deep | a block with k blocks on top can no longer be rolled back; only that state is trusted |
| exchange rate | units of one currency one unit of another buys: here, a currency per ADA |
| spread | an exchange's ask over its bid, minus one: what a round trip costs |
| basis | an exchange's own slow drift from the mid rate, so exchanges disagree slightly |
| cost basis | the average price a wallet paid per ADA, in its primary currency |
| drawdown | a fall from a peak to a later trough; the largest one is the run's worst slide |
| hodl | holding on through a drawdown in the hope rates recover; the hodl agenda never sells at a loss |
| primary currency | the one currency a wallet thinks in; most have no other |
| human | a simulated person; it owns one or more wallets and holds the motivation, temperament and money |
| wallet | a set of keys and the coins they control: what the chain sees |
| owner | the human a wallet belongs to |
| purpose | what a wallet is for to its owner (savings, spending, staking, governance, trading); it limits what the wallet does |
| motivation | the human's agenda; every wallet it owns scores its choices with it |
| wealth tier | low, mid, high or whale: sets pay, bills, savings and how many wallets a human keeps |
| fiat | ordinary money outside the chain, in the human's own currency |
| disposable income | pay less bills in a month, never below zero |
| DCA | dollar-cost averaging: buying a fixed amount on a schedule, whatever the price |
| buffer | months of bills a human keeps in fiat before it invests |
| forced sale | selling ADA to pay bills the fiat could not cover |
| address clustering | guessing which addresses share an owner from on-chain behaviour |
| delegation spread | how far one owner spreads its wallets' stake over different pools |
| synthetic | made up to be plausible, not measured from data |

More: [Seeing Theory](https://seeing-theory.brown.edu/) · [Monte Carlo methods](https://en.wikipedia.org/wiki/Monte_Carlo_method) · [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html) · [scikit-learn mixtures](https://scikit-learn.org/stable/modules/mixture.html)